In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import json

In [ ]:
import torch.nn.functional as F

def train_step(batch, moe_model, optimizer):
    # 1. Get Inputs
    # z_mira: [B, 384], z_text: [B, 3072]
    z_mira = batch['mira_latent'].cuda()
    z_text = batch['graph_embeddings'].cuda()
    target_risk = batch['label'].cuda() # Sepsis or not

    # 2. Forward Pass
    # risk_pred: [B, 1], weights: [B, 3] (The expert weights)
    risk_pred, _, weights = moe_model(z_mira, z_text)

    # 3. COMPUTE LOSSES
    # A. Task Loss: Did we predict Sepsis correctly?
    loss_risk = F.binary_cross_entropy(risk_pred, target_risk)

    # B. Load Balancing Loss: Prevents one expert from taking over everything
    # We want the average weights across a batch to be roughly equal
    # so the model is forced to explore all experts during early training.
    loss_balance = torch.var(weights.mean(dim=0))

    # C. Semantic Alignment (The "Expert Priority" logic)
    # If the itemid belongs to 'Hemodynamic', we can add a small penalty 
    # if the Hemodynamic Expert weight is low.
    loss_total = loss_risk + (0.1 * loss_balance)

    # 4. Backprop
    loss_total.backward()
    optimizer.step()
    optimizer.zero_grad()
    
    return loss_total.item()